# Tarea 1 — Caracterización espectral y prototipado de filtros

CE 1110 — Análisis de Señales Mixtas · Instituto Tecnológico de Costa Rica

Este notebook regenera desde cero las señales de los tres casos de prueba, las
figuras y las métricas del informe. No depende de ningún archivo que no esté en
el repositorio: todos los parámetros salen de `metadatos/configuraciones.json`
y toda la aleatoriedad está fijada por las semillas que ahí se declaran.

Orden de ejecución: de arriba hacia abajo, una sola vez.

## 0. Entorno y configuración

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

import dft_fft
import graficas
import metricas
import senales

COMUN, CASOS = senales.cargar_configuracion()
CARPETA_FIGURAS = RAIZ / 'resultados' / 'figuras'

print(f"fs = {COMUN['fs_hz']} Hz | N total = {COMUN['muestras']} | bloque = {COMUN['n_bloque']}")
print(f"resolucion por bloque: {COMUN['fs_hz']/COMUN['n_bloque']:.2f} Hz | "
      f"latencia: {1000*COMUN['n_bloque']/COMUN['fs_hz']:.1f} ms")
print('casos:', list(CASOS))

## 1. Fundamentos, muestreo y FFT

Apartado 2.1 del enunciado. Relación entre serie de Fourier, transformada de
Fourier, DFT y FFT; resolución espectral y su compromiso con latencia y memoria;
fuga espectral con ventana rectangular y Hann; simetría conjugada.

*Pendiente — bloque 2.*

## 2. Generación y caracterización de señales

Apartado 2.2. Tres casos reproducibles, cada uno con su semilla y sus archivos
WAV de señal limpia, ruido y señal contaminada. El desarrollo completo está en
`docs/2_2_senales.md`.

### 2.a Parámetros de cada caso

In [ ]:
encabezado = f"{'caso':8s} {'semilla':>10s} {'ruido':>8s} {'util (Hz)':>22s} {'SNR obj':>8s}"
print(encabezado); print('-'*len(encabezado))
for identificador, caso in CASOS.items():
    utiles = ', '.join(str(f) for f in caso['frecuencias_utiles_hz'])
    print(f"{identificador:8s} {caso['semilla']:10d} {caso['tipo_ruido']:>8s} "
          f"{utiles:>22s} {caso['snr_objetivo_db']:7.1f} dB")

### 2.b Generación y verificación de la SNR

La SNR se verifica dos veces: en punto flotante y releyendo los WAV ya cuantizados a 16 bits.

In [ ]:
resumen = senales.generar_todos(verbose=False)

print(f"{'caso':8s} {'objetivo':>10s} {'flotante':>10s} {'desde WAV':>10s} {'diferencia':>11s}")
print('-'*53)
for descripcion in resumen:
    diferencia = descripcion['snr_tras_cuantizar_db'] - descripcion['snr_objetivo_db']
    print(f"{descripcion['id']:8s} {descripcion['snr_objetivo_db']:9.4f}  "
          f"{descripcion['snr_obtenida_db']:9.4f}  "
          f"{descripcion['snr_tras_cuantizar_db']:9.4f}  {diferencia:10.4f}")

### 2.c Distribución de energía por bandas

In [ ]:
BORDES = graficas.BORDES_DE_BANDA
etiquetas = [f'{a}-{b}' for a, b in zip(BORDES[:-1], BORDES[1:])]
print(f"{'caso':8s} " + ' '.join(f'{e:>9s}' for e in etiquetas))
print('-'*(9 + 10*len(etiquetas)))
for identificador in CASOS:
    datos = senales.cargar_caso(identificador)
    reparto = metricas.energia_por_bandas(datos['contaminada'], datos['fs'], BORDES)
    print(f'{identificador:8s} ' + ' '.join(f"{100*r['fraccion']:8.1f}%" for r in reparto))

### 2.d Características que permiten identificar el ruido

Planitud espectral y razón pico/mediana. Son las dos que el nodo espectral puede medir solo para escoger su estrategia.

In [ ]:
def planitud_espectral(x):
    densidad = np.abs(np.fft.rfft(x*np.hanning(x.size)))**2 + 1e-20
    return float(np.exp(np.mean(np.log(densidad)))/np.mean(densidad))

def pico_sobre_mediana_db(x):
    densidad = np.abs(np.fft.rfft(x*np.hanning(x.size)))**2 + 1e-20
    return float(10*np.log10(densidad.max()/np.median(densidad)))

print(f"{'caso':8s} {'planitud ruido':>15s} {'planitud cont.':>15s} {'pico/mediana':>14s}")
print('-'*55)
for identificador in CASOS:
    datos = senales.cargar_caso(identificador)
    print(f'{identificador:8s} {planitud_espectral(datos["ruido"]):15.2e} '
          f'{planitud_espectral(datos["contaminada"]):15.2e} '
          f'{pico_sobre_mediana_db(datos["contaminada"]):11.1f} dB')

### 2.e Qué información no se puede recuperar

SNR medida tono por tono: cuánta energía de ruido cae encima de cada componente útil.

In [ ]:
for identificador in CASOS:
    datos = senales.cargar_caso(identificador)
    linea = []
    for f0 in datos['config']['frecuencias_utiles_hz']:
        banda = (f0 - 60, f0 + 60)
        util = metricas.energia_en_banda(datos['limpia'], datos['fs'], banda)
        ruido = metricas.energia_en_banda(datos['ruido'], datos['fs'], banda)
        snr = 10*np.log10(util/(ruido + 1e-20))
        linea.append(f'{f0:5d} Hz: ' + (f'{snr:6.1f} dB' if snr < 80 else '  limpio'))
    print(f'{identificador:8s} | ' + ' | '.join(linea))

### 2.f Figuras

In [ ]:
rutas = graficas.generar_todas(verbose=False)
for ruta in rutas:
    display(Image(filename=str(ruta)))

### 2.g Pruebas automáticas

In [ ]:
sys.path.insert(0, str(RAIZ / 'pruebas'))
import test_senales

test_senales.main()

## 3. Filtro temporal FIR o IIR

Apartado 2.3. Especificación de bandas, justificación del tipo y orden,
coeficientes, `H(z)`, polos y ceros, región de convergencia, estabilidad,
respuesta en magnitud, fase y retardo de grupo, y verificación de que la
ecuación de diferencias propia reproduce la salida de la biblioteca.

*Pendiente — bloque 3.*

## 4. Reducción espectral mediante FFT e IFFT

Apartado 2.4. Procesamiento por bloques de longitud potencia de dos con ventana,
FFT, estimación y modificación espectral, IFFT y reconstrucción con solapamiento.

*Pendiente — bloque 4.*

## 5. Comparación y propuesta para el sistema embebido

Apartado 2.5. Se comparan tres señales contra la referencia limpia: la
contaminada sin filtrar, la salida del filtro temporal y la salida del método
espectral. Antes de medir se compensa el retardo de cada método, estimado por
correlación cruzada, para que el MSE refleje distorsión y no retardo de grupo.

Las salidas las dejan los bloques 3 y 4 en `resultados/salidas/` con los nombres
`caso_N_temporal.wav` y `caso_N_espectral.wav`. Mientras falte alguna, la celda
reporta cuáles y arma la tabla con lo que haya.

In [ ]:
import comparacion

filas = comparacion.main()

## 6. Exportación de resultados

`comparacion.main()` ya escribió `resultados/metricas.csv` y su versión alineada
en `resultados/metricas.txt`. Aquí solo se confirma qué quedó en el disco.

In [ ]:
for carpeta, patron in [('resultados', 'metricas.*'),
                       ('resultados/figuras', '*.png'),
                       ('resultados/salidas', '*.wav')]:
    archivos = sorted((RAIZ / carpeta).glob(patron))
    print(f'{carpeta}/{patron}: {len(archivos)} archivo(s)')
    for archivo in archivos:
        print(f'   {archivo.name:28s} {archivo.stat().st_size/1024:8.1f} KB')